# 05 — Multi-Task Learning: 공유 Backbone + 4-Head 동시 진단

**목표**: 4개 컴포넌트(cooler, valve, pump, accumulator)를 **하나의 모델**로 동시에 진단

**핵심 아이디어 (H4 가설)**:
- 결함 간 상관관계 존재 (예: cooler 고장 → 온도 상승 → accumulator 압력 변화)
- 공유 backbone이 공통 표현을 학습하고, 개별 head가 타깃별 특화
- 독립 학습(04_pytorch_baseline) 대비 성능 향상 기대

**구성**:
1. 데이터 준비 (04와 동일한 전처리)
2. Multi-Task 1D-CNN 모델
3. Multi-Task Loss (weighted sum)
4. 학습 & 평가
5. 독립 학습 vs Multi-Task 비교

## 0. 환경 설정 (Colab)

In [ ]:
import sys, os

if "google.colab" in sys.modules:
    REPO = "/content/hydraulic-phm-internship"
    if not os.path.exists(REPO):
        !git clone https://github.com/pjtae1026-blip/hydraulic-phm-internship.git
    os.chdir(REPO)
    !pip install -q pywavelets scikit-fda umap-learn
    if not os.path.exists("data/uci/PS1.txt"):
        !python scripts/download_data.py

sys.path.insert(0, ".")
print(f"Working dir: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 1. 데이터 로드 & 전처리

04와 동일한 파이프라인: 17개 센서 → 60 timesteps 다운샘플링 → z-score 정규화  
다만 이번에는 **4개 타깃 라벨을 동시에** 사용합니다.

In [ ]:
from src.data_loader import load_all_sensors, load_labels, SENSOR_SPECS

data_dir = __import__("pathlib").Path("data/uci")
sensors = load_all_sensors(data_dir)
labels = load_labels(data_dir)

TARGET_LEN = 60
TARGETS = ["cooler", "valve", "pump", "accumulator"]

def build_tensor(sensors, target_len=TARGET_LEN):
    arrays = []
    for name in SENSOR_SPECS:
        X = sensors[name]
        n_cycles, n_cols = X.shape
        if n_cols > target_len:
            idx = np.linspace(0, n_cols - 1, target_len, dtype=int)
            X = X[:, idx]
        mu, std = X.mean(), X.std()
        if std > 0:
            X = (X - mu) / std
        arrays.append(X)
    return np.stack(arrays, axis=1)

X_all = build_tensor(sensors)
print(f"Input tensor: {X_all.shape}")  # (2205, 17, 60)

In [ ]:
# 각 타깃의 LabelEncoder 및 클래스 수
label_encoders = {}
n_classes_dict = {}
y_encoded = {}

for t in TARGETS:
    le = LabelEncoder()
    y_encoded[t] = le.fit_transform(labels[t].values)
    label_encoders[t] = le
    n_classes_dict[t] = len(le.classes_)
    print(f"{t}: {n_classes_dict[t]} classes → {list(le.classes_)}")

# Train / Test split (동일한 인덱스 사용)
indices = np.arange(len(X_all))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
print(f"\nTrain: {len(train_idx)}, Test: {len(test_idx)}")

## 2. Multi-Task Dataset & DataLoader

In [ ]:
class MultiTaskDataset(Dataset):
    def __init__(self, X, y_dict, indices):
        self.X = torch.FloatTensor(X[indices])
        self.labels = {t: torch.LongTensor(y_dict[t][indices]) for t in y_dict}

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], {t: self.labels[t][idx] for t in self.labels}

BATCH_SIZE = 64

train_ds = MultiTaskDataset(X_all, y_encoded, train_idx)
test_ds = MultiTaskDataset(X_all, y_encoded, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# 확인
X_sample, y_sample = next(iter(train_loader))
print(f"X batch: {X_sample.shape}")
for t in TARGETS:
    print(f"  {t} labels: {y_sample[t].shape}, unique: {y_sample[t].unique().tolist()}")

## 3. Multi-Task 1D-CNN 모델

```
Input (batch, 17, 60)
        │
   ┌────┴────┐
   │ Shared  │  Conv1d×3 blocks (동일 구조)
   │Backbone │  → 공통 feature 추출
   └────┬────┘
        │ (batch, 128)  ← GAP
   ┌────┼────┬────┐
   │    │    │    │
 cooler valve pump accum   ← 독립 FC head
 (3cls) (4cls)(3cls)(4cls)
```

In [ ]:
class MultiTaskCNN(nn.Module):
    def __init__(self, n_channels=17, n_classes_dict=None):
        super().__init__()

        # 공유 Backbone (04와 동일 구조)
        self.backbone = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )
        self.gap = nn.AdaptiveAvgPool1d(1)

        # 타깃별 독립 classification head
        self.heads = nn.ModuleDict()
        for name, nc in n_classes_dict.items():
            self.heads[name] = nn.Sequential(
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, nc),
            )

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.gap(feat).squeeze(-1)  # (batch, 128)
        return {name: head(feat) for name, head in self.heads.items()}

model = MultiTaskCNN(n_channels=17, n_classes_dict=n_classes_dict).to(device)

# 구조 확인
dummy = torch.randn(2, 17, 60).to(device)
outputs = model(dummy)
print(f"Input: {dummy.shape}")
for name, out in outputs.items():
    print(f"  {name} output: {out.shape}")

total_params = sum(p.numel() for p in model.parameters())
backbone_params = sum(p.numel() for p in model.backbone.parameters())
head_params = total_params - backbone_params - sum(p.numel() for p in model.gap.parameters())
print(f"\nTotal params: {total_params:,}")
print(f"  Backbone: {backbone_params:,}")
print(f"  Heads: {head_params:,}")
del dummy, outputs

## 4. Multi-Task Loss & 학습

각 타깃의 CrossEntropyLoss를 가중합산합니다.  
클래스 수가 다르므로 균등 가중치(1/4)로 시작하고, 이후 Uncertainty Weighting도 시도합니다.

In [ ]:
class MultiTaskLoss(nn.Module):
    """Learnable uncertainty-based multi-task loss (Kendall et al., 2018).
    log_sigma^2 per task → loss_t / (2 * sigma_t^2) + log(sigma_t)
    """
    def __init__(self, task_names):
        super().__init__()
        self.task_names = task_names
        self.criteria = {t: nn.CrossEntropyLoss() for t in task_names}
        self.log_vars = nn.ParameterDict({
            t: nn.Parameter(torch.zeros(1)) for t in task_names
        })

    def forward(self, outputs, targets):
        total_loss = 0
        losses = {}
        for t in self.task_names:
            ce = self.criteria[t](outputs[t], targets[t])
            precision = torch.exp(-self.log_vars[t])
            task_loss = precision * ce + self.log_vars[t]
            total_loss += task_loss
            losses[t] = ce.item()
        return total_loss.squeeze(), losses

In [ ]:
def train_one_epoch(model, loader, mt_loss, optimizer):
    model.train()
    total_loss = 0
    task_losses = {t: 0 for t in TARGETS}
    task_correct = {t: 0 for t in TARGETS}
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = {t: y_batch[t].to(device) for t in TARGETS}

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss, losses = mt_loss(outputs, y_batch)
        loss.backward()
        optimizer.step()

        bs = X_batch.size(0)
        total_loss += loss.item() * bs
        total += bs
        for t in TARGETS:
            task_losses[t] += losses[t] * bs
            task_correct[t] += (outputs[t].argmax(1) == y_batch[t]).sum().item()

    n = total
    return (
        total_loss / n,
        {t: task_losses[t] / n for t in TARGETS},
        {t: task_correct[t] / n for t in TARGETS},
    )


@torch.no_grad()
def evaluate(model, loader, mt_loss):
    model.eval()
    total_loss = 0
    task_correct = {t: 0 for t in TARGETS}
    all_preds = {t: [] for t in TARGETS}
    all_labels = {t: [] for t in TARGETS}
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = {t: y_batch[t].to(device) for t in TARGETS}

        outputs = model(X_batch)
        loss, _ = mt_loss(outputs, y_batch)

        bs = X_batch.size(0)
        total_loss += loss.item() * bs
        total += bs
        for t in TARGETS:
            preds = outputs[t].argmax(1)
            task_correct[t] += (preds == y_batch[t]).sum().item()
            all_preds[t].extend(preds.cpu().numpy())
            all_labels[t].extend(y_batch[t].cpu().numpy())

    n = total
    return (
        total_loss / n,
        {t: task_correct[t] / n for t in TARGETS},
        {t: np.array(all_preds[t]) for t in TARGETS},
        {t: np.array(all_labels[t]) for t in TARGETS},
    )

## 5. 학습 실행

In [ ]:
N_EPOCHS = 80
LR = 1e-3

model = MultiTaskCNN(n_channels=17, n_classes_dict=n_classes_dict).to(device)
mt_loss = MultiTaskLoss(TARGETS).to(device)

# 모델 파라미터 + loss의 학습 가능 파라미터(log_vars)도 함께 최적화
optimizer = optim.Adam(
    list(model.parameters()) + list(mt_loss.parameters()),
    lr=LR, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

history = {
    "total_loss": [], "val_total_loss": [],
    **{f"train_acc_{t}": [] for t in TARGETS},
    **{f"val_acc_{t}": [] for t in TARGETS},
    "task_weights": [],
}

best_avg_acc = 0
best_state = None

for epoch in range(N_EPOCHS):
    train_loss, train_task_losses, train_accs = train_one_epoch(model, train_loader, mt_loss, optimizer)
    val_loss, val_accs, _, _ = evaluate(model, test_loader, mt_loss)
    scheduler.step()

    history["total_loss"].append(train_loss)
    history["val_total_loss"].append(val_loss)
    for t in TARGETS:
        history[f"train_acc_{t}"].append(train_accs[t])
        history[f"val_acc_{t}"].append(val_accs[t])

    # 학습된 task weights 기록
    weights = {t: torch.exp(-mt_loss.log_vars[t]).item() for t in TARGETS}
    history["task_weights"].append(weights)

    avg_val_acc = np.mean([val_accs[t] for t in TARGETS])
    if avg_val_acc > best_avg_acc:
        best_avg_acc = avg_val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if (epoch + 1) % 10 == 0:
        acc_str = " | ".join(f"{t}: {val_accs[t]:.3f}" for t in TARGETS)
        w_str = " | ".join(f"{t}: {weights[t]:.2f}" for t in TARGETS)
        print(f"Epoch {epoch+1:3d}/{N_EPOCHS} | Val Acc [{acc_str}] | Avg: {avg_val_acc:.4f}")
        print(f"  Task weights: [{w_str}]")

print(f"\nBest avg val accuracy: {best_avg_acc:.4f}")

## 6. 최종 평가 (Best Model)

In [ ]:
# Best 모델 로드 & 최종 평가
model.load_state_dict(best_state)
model.to(device)

_, final_accs, final_preds, final_labels = evaluate(model, test_loader, mt_loss)

for t in TARGETS:
    print(f"\n{'='*50}")
    print(f"  {t.upper()} — Test Accuracy: {final_accs[t]:.4f}")
    print(f"{'='*50}")
    target_names = [str(c) for c in label_encoders[t].classes_]
    print(classification_report(final_labels[t], final_preds[t], target_names=target_names))

## 7. 시각화

### 7.1 학습 곡선 (타깃별 Accuracy)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
epochs = range(1, N_EPOCHS + 1)

for i, t in enumerate(TARGETS):
    axes[i].plot(epochs, history[f"train_acc_{t}"], label="Train", alpha=0.8)
    axes[i].plot(epochs, history[f"val_acc_{t}"], label="Val", alpha=0.8)
    axes[i].set_title(f"{t}")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Accuracy")
    axes[i].set_ylim(0, 1.05)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle("Multi-Task 1D-CNN — Per-Task Accuracy", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("reports/05_multitask_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 학습된 Task Weights 변화

Uncertainty weighting이 각 타깃에 어떤 가중치를 부여하는지 확인합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
epochs = range(1, N_EPOCHS + 1)

for t in TARGETS:
    ws = [history["task_weights"][e][t] for e in range(N_EPOCHS)]
    ax.plot(epochs, ws, label=t, linewidth=2)

ax.set_xlabel("Epoch")
ax.set_ylabel("Task Weight (precision = 1/σ²)")
ax.set_title("Learned Task Weights over Training")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("reports/05_task_weights.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.3 Confusion Matrix (Multi-Task)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for i, t in enumerate(TARGETS):
    cm = confusion_matrix(final_labels[t], final_preds[t])
    class_names = [str(c) for c in label_encoders[t].classes_]

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[i])
    axes[i].set_title(f"{t} (acc: {final_accs[t]:.3f})")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("True")

plt.suptitle("Confusion Matrices — Multi-Task 1D-CNN", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("reports/05_multitask_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. 독립 학습 vs Multi-Task 비교

04_pytorch_baseline의 결과와 비교합니다.  
**아래 셀의 `single_task_acc`를 04 노트북 실행 결과로 업데이트하세요.**

In [ ]:
# ⚠️ 04 노트북 실행 후 아래 값을 실제 결과로 교체하세요
single_task_acc = {
    "cooler": 0.0,       # 04 결과로 교체
    "valve": 0.0,        # 04 결과로 교체
    "pump": 0.0,         # 04 결과로 교체
    "accumulator": 0.0,  # 04 결과로 교체
}

multi_task_acc = {t: final_accs[t] for t in TARGETS}

comparison = []
for t in TARGETS:
    st = single_task_acc[t]
    mt = multi_task_acc[t]
    diff = mt - st
    comparison.append({
        "Target": t,
        "Single-Task": f"{st:.4f}" if st > 0 else "TBD",
        "Multi-Task": f"{mt:.4f}",
        "Δ (MT - ST)": f"{diff:+.4f}" if st > 0 else "—",
    })

comp_df = pd.DataFrame(comparison)
print("=" * 60)
print("  Independent (Single-Task) vs Multi-Task Comparison")
print("=" * 60)
print(comp_df.to_string(index=False))
print("=" * 60)

if all(v > 0 for v in single_task_acc.values()):
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(TARGETS))
    w = 0.35
    ax.bar(x - w/2, [single_task_acc[t] for t in TARGETS], w, label="Single-Task", color="steelblue")
    ax.bar(x + w/2, [multi_task_acc[t] for t in TARGETS], w, label="Multi-Task", color="coral")
    ax.set_xticks(x)
    ax.set_xticklabels(TARGETS)
    ax.set_ylabel("Test Accuracy")
    ax.set_ylim(0, 1.05)
    ax.set_title("Single-Task vs Multi-Task 1D-CNN")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig("reports/05_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("\n⚠️ 04 노트북 결과를 입력한 후 이 셀을 다시 실행하면 비교 그래프가 생성됩니다.")

## 9. 분석 & 결론

### Multi-Task Learning의 장점
- **파라미터 효율성**: backbone 공유로 독립 모델 4개 대비 파라미터 대폭 절약
- **암묵적 정규화**: 여러 타깃을 동시에 학습하면 과적합 억제 효과
- **결함 간 상관 학습**: 공유 표현이 컴포넌트 간 물리적 연결을 반영

### Uncertainty Weighting 효과
- 학습 난이도가 다른 타깃의 loss를 자동으로 균형 조정
- 어려운 타깃(valve, pump)의 가중치가 자동 조절되는지 확인

### H4 가설 검증
- Multi-task가 single-task 대비 성능이 향상되면 H4 가설 지지
- 특히 상관관계가 강한 타깃 쌍에서 큰 개선 기대

### Next: Week 4 — 본인 가설 검증
- `07_my_model.ipynb`: 압력센서 cross-correlation (H1) + Transformer attention (H5) 등